# Ejercicio 11: Web Scraping

### Nombre: Anthony Goyes

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

## Parte 0: Planificación

Para esta práctica se trabajará con una página HTML de una receta de cocina. El objetivo es extraer información estructurada desde el HTML y construir un pequeño corpus de recetas.

**Sitio objetivo:** página de receta almacenada localmente como archivo HTML.  
**Archivo usado:** `rotisserie-chicken.html`

**Datos a extraer:**
- Título de la receta
- Descripción
- Ingredientes
- Instrucciones
- Información nutricional
- Enlaces relacionados a otras recetas

**Estructura del corpus:**
Cada receta se almacenará como un documento con los siguientes campos:

```json
{
  "title": "Título de la receta",
  "description": "Descripción",
  "ingredients": ["ingrediente 1", "ingrediente 2"],
  "instructions": ["paso 1", "paso 2"],
  "nutrition_facts": ["calorías", "proteínas", "grasas"],
  "source_file": "archivo HTML original"
}

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [1]:
from bs4 import BeautifulSoup
from pathlib import Path

html_path = Path("../data/rotisserie-chicken.html")

print("Existe el archivo:", html_path.exists())
print("Ruta:", html_path.resolve())

Existe el archivo: True
Ruta: D:\Septimo\RI\ir25b\data\rotisserie-chicken.html


In [2]:
with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, "html.parser")

print("HTML cargado correctamente")
print("Tamaño del HTML:", len(html_content), "caracteres")

HTML cargado correctamente
Tamaño del HTML: 579302 caracteres


In [3]:
def get_meta_content(soup, attrs):
    tag = soup.find("meta", attrs)
    if tag and tag.get("content"):
        return tag["content"].strip()
    return None

title = get_meta_content(soup, {"property": "og:title"})

if title is None:
    title = page_title

title

'Rotisserie Chicken'

In [4]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")

for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

Aquí vamos a extraer descripción, ingredientes, instrucciones y nutrición.

### Descripción

In [5]:
description = get_meta_content(soup, {"name": "description"})

description

"Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves."

### Ingredientes

In [6]:
ingredients_section = soup.find_all(
    "li",
    class_="mm-recipes-structured-ingredients__list-item"
)

ingredients = [
    ingredient.get_text(" ", strip=True)
    for ingredient in ingredients_section
]

print("Total de ingredientes:", len(ingredients))

for ingredient in ingredients:
    print("-", ingredient)

Total de ingredientes: 6
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper


### Instrucciones

In [7]:
instructions_section = soup.find_all(
    "p",
    class_="comp mntl-sc-block mntl-sc-block-html"
)

instructions = [
    instruction.get_text(" ", strip=True)
    for instruction in instructions_section
]

instructions = [
    instruction
    for instruction in instructions
    if len(instruction) > 20
]

print("Total de instrucciones/textos encontrados:", len(instructions))

for i, instruction in enumerate(instructions, start=1):
    print(f"{i}. {instruction}")

Total de instrucciones/textos encontrados: 23
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken : This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time. · Butter : Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to. · Seasonings : The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — but here's a brief overview of what you can expect when you make this rotisserie chicken:
5. Season the chicken cavity with salt and place it on a rotisserie (tie the wings and legs if you have twine or string available). Set the grill on high and cook for 10 minutes. Turn the grill down to medium and baste with a mixture

### Información nutricional

In [8]:
nutrition_section = soup.find_all(
    "span",
    class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix"
)

nutrition_facts = [
    fact.parent.get_text(" ", strip=True)
    for fact in nutrition_section
    if fact.parent
]

print("Total de datos nutricionales:", len(nutrition_facts))

for fact in nutrition_facts:
    print("-", fact)

Total de datos nutricionales: 12
- Total Fat 25g
- Saturated Fat 10g
- Cholesterol 117mg
- Sodium 1311mg
- Total Carbohydrate 1g
- Dietary Fiber 1g
- Total Sugars 0g
- Protein 31g
- Vitamin C 1mg
- Calcium 22mg
- Iron 2mg
- Potassium 302mg


### Crear un diccionario con la receta

In [9]:
recipe_data = {
    "title": title,
    "description": description,
    "ingredients": ingredients,
    "instructions": instructions,
    "nutrition_facts": nutrition_facts,
    "source_file": str(html_path)
}

recipe_data

{'title': 'Rotisserie Chicken',
 'description': "Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.",
 'ingredients': ['1 (3 pound) whole chicken',
  '1 pinch salt',
  '¼ cup butter, melted',
  '1 tablespoon salt',
  '1 tablespoon ground paprika',
  '¼ tablespoon ground black pepper'],
 'instructions': ["Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.",
  "Here's what you'll need to make rotisserie chicken at home:",
  "· Whole Chicken : This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time. · Butter : Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to. · Seasonings : The rotisserie chicken is simply seasoned with salt, pepper, and paprika.",
  "You'll find 

### Mostrar todo ordenado

In [10]:
print("Recipe Title:", recipe_data["title"])
print()
print("Description:", recipe_data["description"])
print()

print("Ingredients:")
for ingredient in recipe_data["ingredients"]:
    print("-", ingredient)

print()
print("Instructions:")
for i, instruction in enumerate(recipe_data["instructions"], start=1):
    print(f"{i}. {instruction}")

print()
print("Nutrition Facts:")
for fact in recipe_data["nutrition_facts"]:
    print("-", fact)

Recipe Title: Rotisserie Chicken

Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.

Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper

Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken : This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time. · Butter : Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to. · Seasonings : The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [11]:
recipe_links = soup.find_all("a", href=True)

recipe_urls = []

for link in recipe_links:
    href = link["href"]
    
    if "recipe" in href.lower():
        recipe_urls.append(href)

recipe_urls = list(dict.fromkeys(recipe_urls))

print("Total de enlaces relacionados:", len(recipe_urls))

for url in recipe_urls[:20]:
    print(url)

Total de enlaces relacionados: 113
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com

In [12]:
from urllib.parse import urljoin

base_url = "https://www.allrecipes.com/"

recipe_urls_absolute = [
    urljoin(base_url, url)
    for url in recipe_urls
]

recipe_urls_absolute = list(dict.fromkeys(recipe_urls_absolute))

print("Total de enlaces absolutos:", len(recipe_urls_absolute))

for url in recipe_urls_absolute[:20]:
    print(url)

Total de enlaces absolutos: 113
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.allrecipes.com/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
htt

In [13]:
import re
from urllib.parse import urlparse

def is_recipe_detail_url(url):
    parsed = urlparse(url)
    
    # Debe ser del dominio allrecipes
    if "allrecipes.com" not in parsed.netloc:
        return False
    
    # Debe tener estructura /recipe/NUMERO/nombre-receta/
    return re.search(r"^/recipe/\d+/", parsed.path) is not None

In [14]:
recipe_detail_urls = [
    url for url in recipe_urls_absolute
    if is_recipe_detail_url(url)
]

# Quitar repetidos
recipe_detail_urls = list(dict.fromkeys(recipe_detail_urls))

print("Total de URLs de recetas reales:", len(recipe_detail_urls))

for url in recipe_detail_urls[:20]:
    print(url)

Total de URLs de recetas reales: 16
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
https://www.allrecipes.com/recipe/

### Crear función para extraer una receta

In [15]:
def get_meta_content(soup, attrs):
    tag = soup.find("meta", attrs)
    
    if tag and tag.get("content"):
        return tag["content"].strip()
    
    return None

In [16]:
def parse_recipe_from_soup(soup, source):
    # Título
    title = get_meta_content(soup, {"property": "og:title"})
    
    if title is None and soup.title:
        title = soup.title.get_text(strip=True)
    
    # Descripción
    description = get_meta_content(soup, {"name": "description"})
    
    # Ingredientes
    ingredients_section = soup.find_all(
        "li",
        class_="mm-recipes-structured-ingredients__list-item"
    )
    
    ingredients = [
        ingredient.get_text(" ", strip=True)
        for ingredient in ingredients_section
    ]
    
    # Instrucciones
    instructions_section = soup.find_all(
        "p",
        class_="comp mntl-sc-block mntl-sc-block-html"
    )
    
    instructions = [
        instruction.get_text(" ", strip=True)
        for instruction in instructions_section
    ]
    
    # Limpieza básica de instrucciones
    instructions = [
        instruction
        for instruction in instructions
        if len(instruction) > 20
    ]
    
    # Eliminar instrucciones repetidas manteniendo orden
    instructions = list(dict.fromkeys(instructions))
    
    # Información nutricional
    nutrition_section = soup.find_all(
        "span",
        class_="mm-recipes-nutrition-facts-label__nutrient-name"
    )
    
    nutrition_facts = []
    
    for nutrient in nutrition_section:
        if nutrient.parent:
            text = nutrient.parent.get_text(" ", strip=True)
            nutrition_facts.append(text)
    
    nutrition_facts = list(dict.fromkeys(nutrition_facts))
    
    recipe = {
        "title": title,
        "description": description,
        "ingredients": ingredients,
        "instructions": instructions,
        "nutrition_facts": nutrition_facts,
        "source": source
    }
    
    return recipe

In [17]:
import requests
import time
from bs4 import BeautifulSoup

def fetch_url(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()
    
    return response.text

In [18]:
main_recipe = parse_recipe_from_soup(
    soup,
    source=str(html_path)
)

print("Receta principal:", main_recipe["title"])
print("Ingredientes:", len(main_recipe["ingredients"]))
print("Instrucciones:", len(main_recipe["instructions"]))

Receta principal: Rotisserie Chicken
Ingredientes: 6
Instrucciones: 23


### Crear el corpus con tu receta local + las 16 recetas

In [19]:
MAX_RELATED_RECIPES = 16

recipes = [main_recipe]
seen_titles = {main_recipe["title"]}

for url in recipe_detail_urls[:MAX_RELATED_RECIPES]:
    try:
        print("Descargando receta:", url)
        
        html = fetch_url(url)
        related_soup = BeautifulSoup(html, "html.parser")
        
        recipe = parse_recipe_from_soup(
            related_soup,
            source=url
        )
        
        if (
            recipe["title"] 
            and len(recipe["ingredients"]) > 0 
            and recipe["title"] not in seen_titles
        ):
            recipes.append(recipe)
            seen_titles.add(recipe["title"])
            print("OK:", recipe["title"], "| Ingredientes:", len(recipe["ingredients"]))
        else:
            print("No se pudo extraer bien o está duplicada:", url)
        
        time.sleep(1)
    
    except Exception as e:
        print("Error con:", url)
        print(e)

Descargando receta: https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
OK: Cilantro-Lime Grilled Chicken | Ingredientes: 5
Descargando receta: https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
OK: Buttermilk Barbecue Chicken | Ingredientes: 11
Descargando receta: https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
OK: Grilled Spatchcocked Chicken | Ingredientes: 8
Descargando receta: https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
OK: Beer Butt Chicken | Ingredientes: 6
Descargando receta: https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
OK: Good Frickin’ Paprika Chicken | Ingredientes: 14
Descargando receta: https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
OK: Miso Honey Chicken | Ingredientes: 9
Descargando receta: https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
OK: Rosemary Buttermilk Chicken | Ingredientes: 6
Descargando receta: https://www.allrecipes.com/r

### Verificar el corpus

In [20]:
print("Total de recetas en el corpus:", len(recipes))

for recipe in recipes:
    print("-", recipe["title"], "| Ingredientes:", len(recipe["ingredients"]))

Total de recetas en el corpus: 17
- Rotisserie Chicken | Ingredientes: 6
- Cilantro-Lime Grilled Chicken | Ingredientes: 5
- Buttermilk Barbecue Chicken | Ingredientes: 11
- Grilled Spatchcocked Chicken | Ingredientes: 8
- Beer Butt Chicken | Ingredientes: 6
- Good Frickin’ Paprika Chicken | Ingredientes: 14
- Miso Honey Chicken | Ingredientes: 9
- Rosemary Buttermilk Chicken | Ingredientes: 6
- Smoked Beer Butt Chicken | Ingredientes: 9
- The Best Beer Can Chicken Ever | Ingredientes: 15
- Best Beer Can Chicken | Ingredientes: 9
- Drunk Chicken | Ingredientes: 6
- Grilled Chicken Under a Brick | Ingredientes: 5
- Smoked Whole Chicken | Ingredientes: 9
- Easy Barbeque Chicken | Ingredientes: 7
- Darn Good Chicken | Ingredientes: 4
- Beer Can Chicken | Ingredientes: 8


### Convertir a DataFrame

In [21]:
import pandas as pd

recipes_df = pd.DataFrame([
    {
        "title": recipe["title"],
        "description": recipe["description"],
        "ingredients": " | ".join(recipe["ingredients"]),
        "instructions": " | ".join(recipe["instructions"]),
        "nutrition_facts": " | ".join(recipe["nutrition_facts"]),
        "source": recipe["source"]
    }
    for recipe in recipes
])

recipes_df

,title,description,ingredients,instructions,nutrition_facts,source
0,Rotisserie Chicken,Rotisserie chicken that's easy to cook on a ga...,1 (3 pound) whole chicken | 1 pinch salt | ¼ c...,Intimidated by the idea of making a rotisserie...,Total Fat 25g | Saturated Fat 10g | Cholestero...,..\data\rotisserie-chicken.html
1,Cilantro-Lime Grilled Chicken,This cilantro-lime grilled chicken recipe star...,"½ cup chopped fresh cilantro | 4 limes, juiced...","Whisk cilantro, lime juice, garlic salt, and b...",Total Carbohydrate 4g | Dietary Fiber 1g | Tot...,https://www.allrecipes.com/recipe/238575/cilan...
2,Buttermilk Barbecue Chicken,Chicken turns out super moist and flavorful on...,2 cups buttermilk | ¼ cup brown sugar | 1 tabl...,"Whisk buttermilk, brown sugar, cider vinegar, ...",Total Carbohydrate 15g | Dietary Fiber 1g | To...,https://www.allrecipes.com/recipe/275062/butte...
3,Grilled Spatchcocked Chicken,This grilled spatchcock chicken recipe calls f...,¼ cup kosher salt | water | 1 (4 pound) whole ...,Place salt in a large bowl or Dutch oven; add ...,Total Carbohydrate 5g | Dietary Fiber 1g | Tot...,https://www.allrecipes.com/recipe/274724/grill...
4,Beer Butt Chicken,"This beer butter chicken recipe combines beer,...","1 cup butter, divided | 2 tablespoons garlic s...",Preheat an outdoor grill for low heat and ligh...,Total Carbohydrate 3g | Dietary Fiber 1g | Tot...,https://www.allrecipes.com/recipe/14531/beer-b...
5,Good Frickin’ Paprika Chicken,Chef John named his delicious yogurt- and papr...,"6 tablespoons plain yogurt | 3 cloves garlic, ...","Whisk together yogurt, garlic, 3 tablespoons p...",Total Carbohydrate 6g | Dietary Fiber 1g | Tot...,https://www.allrecipes.com/recipe/221093/good-...
6,Miso Honey Chicken,Chicken is marinated in Chef John's magical mi...,3 tablespoons white miso | 2 tablespoons honey...,Combine miso and honey in a bowl. Pour in rice...,Total Carbohydrate 7g | Dietary Fiber 1g | Tot...,https://www.allrecipes.com/recipe/264278/miso-...
7,Rosemary Buttermilk Chicken,Summer is the time to master this garlicky but...,"2 ½ cups buttermilk | 15 cloves garlic, minced...","Mix buttermilk, garlic, paprika, salt, and pep...",Total Carbohydrate 8g | Dietary Fiber 1g | Tot...,https://www.allrecipes.com/recipe/258659/rosem...
8,Smoked Beer Butt Chicken,Roast a chicken on your grill with a can of be...,1 (3 pound) whole chicken | ¼ cup vegetable oi...,Preheat grill for medium heat and lightly oil ...,Total Carbohydrate 10g | Dietary Fiber 1g | To...,https://www.allrecipes.com/recipe/222936/smoke...
9,The Best Beer Can Chicken Ever,A whole grilled chicken gets a burst of flavor...,1 cup chocolate stout beer | 3 green Thai chil...,Preheat grill for medium heat. If using charco...,Total Carbohydrate 10g | Dietary Fiber 1g | To...,https://www.allrecipes.com/recipe/228070/the-b...


### Guardar el corpus

In [22]:
import json
from pathlib import Path

output_json = Path("../data/recipes_corpus.json")

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(recipes, f, ensure_ascii=False, indent=2)

print("Corpus guardado en:", output_json)

Corpus guardado en: ..\data\recipes_corpus.json


## Construcción del corpus de recetas

Después de identificar las URLs reales de recetas, se descargó cada página utilizando `requests` y se procesó su contenido HTML con BeautifulSoup.

Para evitar páginas que no fueran recetas, se filtraron únicamente URLs con la estructura `/recipe/{id}/{nombre-receta}/`. Luego, se reutilizó la función `parse_recipe_from_soup()` para extraer de cada receta el título, descripción, ingredientes, instrucciones, datos nutricionales y fuente.

Finalmente, todas las recetas extraídas se almacenaron en una lista de diccionarios y posteriormente se convirtieron a un DataFrame. Este DataFrame representa el corpus estructurado que será usado en la siguiente parte para recuperación de información y RAG.

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

### Cargar el corpus de recetas

In [34]:
import json
from pathlib import Path
import pandas as pd

corpus_path = Path("../data/recipes_corpus.json")

with open(corpus_path, "r", encoding="utf-8") as f:
    recipes = json.load(f)

print("Total de recetas cargadas:", len(recipes))

for recipe in recipes:
    print("-", recipe["title"])

Total de recetas cargadas: 17
- Rotisserie Chicken
- Cilantro-Lime Grilled Chicken
- Buttermilk Barbecue Chicken
- Grilled Spatchcocked Chicken
- Beer Butt Chicken
- Good Frickin’ Paprika Chicken
- Miso Honey Chicken
- Rosemary Buttermilk Chicken
- Smoked Beer Butt Chicken
- The Best Beer Can Chicken Ever
- Best Beer Can Chicken
- Drunk Chicken
- Grilled Chicken Under a Brick
- Smoked Whole Chicken
- Easy Barbeque Chicken
- Darn Good Chicken
- Beer Can Chicken


### Convertir recetas en documentos para RAG

In [35]:
def build_recipe_documents(recipes):
    documents = []
    
    for recipe_id, recipe in enumerate(recipes):
        title = recipe.get("title", "")
        source = recipe.get("source", "")
        
        # Documento de descripción
        if recipe.get("description"):
            documents.append({
                "recipe_id": recipe_id,
                "title": title,
                "section": "description",
                "text": f"Title: {title}\nDescription: {recipe['description']}",
                "source": source
            })
        
        # Documento de ingredientes
        if recipe.get("ingredients"):
            ingredients_text = "\n".join([
                f"- {ingredient}"
                for ingredient in recipe["ingredients"]
            ])
            
            documents.append({
                "recipe_id": recipe_id,
                "title": title,
                "section": "ingredients",
                "text": f"Title: {title}\nIngredients:\n{ingredients_text}",
                "source": source
            })
        
        # Documento de instrucciones
        if recipe.get("instructions"):
            instructions_text = "\n".join([
                f"{idx}. {step}"
                for idx, step in enumerate(recipe["instructions"], start=1)
            ])
            
            documents.append({
                "recipe_id": recipe_id,
                "title": title,
                "section": "instructions",
                "text": f"Title: {title}\nInstructions:\n{instructions_text}",
                "source": source
            })
        
        # Documento de nutrición
        if recipe.get("nutrition_facts"):
            nutrition_text = "\n".join([
                f"- {fact}"
                for fact in recipe["nutrition_facts"]
            ])
            
            documents.append({
                "recipe_id": recipe_id,
                "title": title,
                "section": "nutrition",
                "text": f"Title: {title}\nNutrition Facts:\n{nutrition_text}",
                "source": source
            })
    
    return documents

In [36]:
rag_documents = build_recipe_documents(recipes)

print("Total de documentos para RAG:", len(rag_documents))

rag_documents[:3]

Total de documentos para RAG: 68


[{'recipe_id': 0,
  'title': 'Rotisserie Chicken',
  'section': 'description',
  'text': "Title: Rotisserie Chicken\nDescription: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.",
  'source': '..\\data\\rotisserie-chicken.html'},
 {'recipe_id': 0,
  'title': 'Rotisserie Chicken',
  'section': 'ingredients',
  'text': 'Title: Rotisserie Chicken\nIngredients:\n- 1 (3 pound) whole chicken\n- 1 pinch salt\n- ¼ cup butter, melted\n- 1 tablespoon salt\n- 1 tablespoon ground paprika\n- ¼ tablespoon ground black pepper',
  'source': '..\\data\\rotisserie-chicken.html'},
 {'recipe_id': 0,
  'title': 'Rotisserie Chicken',
  'section': 'instructions',
  'text': 'Title: Rotisserie Chicken\nInstructions:\n1. Intimidated by the idea of making a rotisserie chicken at home? We\'re here to help. Get your grill and rotisserie attachment ready — you\'ll want to try this recipe ASAP.\n2. Here\'s what you\'

In [39]:
rag_df = pd.DataFrame(rag_documents)

rag_df.head()

,recipe_id,title,section,text,source
0,0,Rotisserie Chicken,description,Title: Rotisserie Chicken\nDescription: Rotiss...,..\data\rotisserie-chicken.html
1,0,Rotisserie Chicken,ingredients,Title: Rotisserie Chicken\nIngredients:\n- 1 (...,..\data\rotisserie-chicken.html
2,0,Rotisserie Chicken,instructions,Title: Rotisserie Chicken\nInstructions:\n1. I...,..\data\rotisserie-chicken.html
3,0,Rotisserie Chicken,nutrition,Title: Rotisserie Chicken\nNutrition Facts:\n-...,..\data\rotisserie-chicken.html
4,1,Cilantro-Lime Grilled Chicken,description,Title: Cilantro-Lime Grilled Chicken\nDescript...,https://www.allrecipes.com/recipe/238575/cilan...


In [40]:
rag_df["section"].value_counts()

section
description     17
ingredients     17
instructions    17
nutrition       17
Name: count, dtype: int64

### Crear índice TF-IDF

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(rag_df["text"])

print("Matriz TF-IDF:", tfidf_matrix.shape)

Matriz TF-IDF: (68, 2967)


### Función para recuperar contexto

In [55]:
import numpy as np
import re

def retrieve_recipe_context(query, top_k=5):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, tfidf_matrix)[0]
    
    q = query.lower()
    
    # Copiamos los scores originales
    final_scores = similarities.copy()
    
    # Detectar qué tipo de sección busca la pregunta
    target_section = None
    
    if (
        "how" in q 
        or "cook" in q 
        or "prepare" in q 
        or "instruction" in q 
        or "instructions" in q
        or "steps" in q
    ):
        target_section = "instructions"
    
    elif "ingredient" in q or "ingredients" in q:
        target_section = "ingredients"
    
    elif (
        "nutrition" in q 
        or "calorie" in q 
        or "calories" in q 
        or "protein" in q
    ):
        target_section = "nutrition"
    
    # Dar boost a la sección correcta
    if target_section is not None:
        section_boost = rag_df["section"].apply(
            lambda section: 0.25 if section == target_section else 0
        ).values
        
        final_scores = final_scores + section_boost
    
    # Dar boost si las palabras del título aparecen en la pregunta
    query_words = set(re.findall(r"[a-zA-Z]+", q))
    
    title_boosts = []
    
    for title in rag_df["title"]:
        title_words = set(re.findall(r"[a-zA-Z]+", title.lower()))
        
        if len(title_words) == 0:
            title_boosts.append(0)
        else:
            overlap = len(query_words.intersection(title_words)) / len(title_words)
            title_boosts.append(0.15 * overlap)
    
    final_scores = final_scores + np.array(title_boosts)
    
    top_indices = final_scores.argsort()[::-1][:top_k]
    
    results = []
    
    for rank, idx in enumerate(top_indices, start=1):
        row = rag_df.iloc[idx]
        
        results.append({
            "rank": rank,
            "score": float(final_scores[idx]),
            "base_score": float(similarities[idx]),
            "recipe_id": int(row["recipe_id"]),
            "title": row["title"],
            "section": row["section"],
            "text": row["text"],
            "source": row["source"]
        })
    
    return results

### Probar recuperación sin Gemini

In [56]:
query = "What ingredients are needed for beer can chicken?"

retrieved_results = retrieve_recipe_context(query, top_k=5)

for result in retrieved_results:
    print("=" * 100)
    print("Rank:", result["rank"])
    print("Score:", result["score"])
    print("Title:", result["title"])
    print("Section:", result["section"])
    print()
    print(result["text"][:800])

Rank: 1
Score: 0.6227261079686184
Title: Beer Can Chicken
Section: ingredients

Title: Beer Can Chicken
Ingredients:
- ⅓ cup brown sugar
- 2 tablespoons chili powder
- 2 tablespoons paprika
- 2 teaspoons dry mustard
- ½ teaspoon salt
- ¼ teaspoon ground black pepper
- ½ (12 fluid ounce) can beer
- 1 (3 pound) whole chicken
Rank: 2
Score: 0.5243607555899774
Title: Best Beer Can Chicken
Section: ingredients

Title: Best Beer Can Chicken
Ingredients:
- 2 cups cherry wood chips
- ½ cup dark brown sugar
- ½ cup salt
- ½ cup paprika
- ¼ cup ground black pepper
- 1 teaspoon cayenne pepper
- 2 (12 fluid ounce) cans beer, half full
- 2 (3 pound) whole chickens
- ¼ cup vegetable oil
Rank: 3
Score: 0.4889539514445618
Title: Beer Butt Chicken
Section: ingredients

Title: Beer Butt Chicken
Ingredients:
- 1 cup butter, divided
- 2 tablespoons garlic salt, divided
- 2 tablespoons paprika, divided
- salt and pepper to taste
- 1 (12 fluid ounce) can beer
- 1 (4 pound) whole chicken
Rank: 4
Score: 0.450

In [57]:
retrieved_df = pd.DataFrame([
    {
        "rank": r["rank"],
        "score": r["score"],
        "title": r["title"],
        "section": r["section"],
        "source": r["source"]
    }
    for r in retrieved_results
])

retrieved_df

,rank,score,title,section,source
0,1,0.622726,Beer Can Chicken,ingredients,https://www.allrecipes.com/recipe/214618/beer-...
1,2,0.524361,Best Beer Can Chicken,ingredients,https://www.allrecipes.com/recipe/214619/bbq-b...
2,3,0.488954,Beer Butt Chicken,ingredients,https://www.allrecipes.com/recipe/14531/beer-b...
3,4,0.450395,Smoked Beer Butt Chicken,ingredients,https://www.allrecipes.com/recipe/222936/smoke...
4,5,0.439787,The Best Beer Can Chicken Ever,ingredients,https://www.allrecipes.com/recipe/228070/the-b...


### Configurar Gemini

In [45]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
from pathlib import Path

# Cargar .env desde posibles ubicaciones
load_dotenv()
load_dotenv(Path(".env"))
load_dotenv(Path("../.env"))
load_dotenv(Path("../notebooks/.env"))

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("No se encontró GEMINI_API_KEY. Revisa tu archivo .env.")

client = genai.Client(api_key=api_key)

print("Cliente de Gemini creado correctamente")

Cliente de Gemini creado correctamente


### Construir contexto para Gemini

In [58]:
def build_context_from_results(results):
    context_parts = []
    
    for r in results:
        context_parts.append(
            f"""
Documento {r['rank']}
Receta: {r['title']}
Sección: {r['section']}
Fuente: {r['source']}
Score de recuperación: {r['score']:.4f}

Contenido:
{r['text']}
"""
        )
    
    return "\n\n".join(context_parts)

In [59]:
context = build_context_from_results(retrieved_results)

print(context[:2000])


Documento 1
Receta: Beer Can Chicken
Sección: ingredients
Fuente: https://www.allrecipes.com/recipe/214618/beer-can-chicken/
Score de recuperación: 0.6227

Contenido:
Title: Beer Can Chicken
Ingredients:
- ⅓ cup brown sugar
- 2 tablespoons chili powder
- 2 tablespoons paprika
- 2 teaspoons dry mustard
- ½ teaspoon salt
- ¼ teaspoon ground black pepper
- ½ (12 fluid ounce) can beer
- 1 (3 pound) whole chicken



Documento 2
Receta: Best Beer Can Chicken
Sección: ingredients
Fuente: https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
Score de recuperación: 0.5244

Contenido:
Title: Best Beer Can Chicken
Ingredients:
- 2 cups cherry wood chips
- ½ cup dark brown sugar
- ½ cup salt
- ½ cup paprika
- ¼ cup ground black pepper
- 1 teaspoon cayenne pepper
- 2 (12 fluid ounce) cans beer, half full
- 2 (3 pound) whole chickens
- ¼ cup vegetable oil



Documento 3
Receta: Beer Butt Chicken
Sección: ingredients
Fuente: https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
Score d

### Función RAG con Gemini

In [60]:
def rag_answer_with_gemini(query, top_k=5):
    # 1. Recuperar contexto desde el corpus
    retrieved_results = retrieve_recipe_context(query, top_k=top_k)
    
    # 2. Construir contexto textual
    context = build_context_from_results(retrieved_results)
    
    # 3. Crear prompt para Gemini
    prompt = f"""
Eres un asistente de recetas basado en Recuperación de Información.

El usuario quiere saber:
"{query}"

Mi Sistema de Recuperación de Información encontró este contexto en el corpus de recetas:

{context}

Instrucciones importantes:
1. Responde únicamente usando la información del contexto recuperado.
2. No inventes ingredientes, pasos, tiempos ni datos nutricionales.
3. Si el contexto no contiene suficiente información, responde:
"No tengo suficiente información en el corpus para responder."
4. Responde de forma clara y útil.
5. Si mencionas una receta, usa el nombre exacto que aparece en el contexto.
6. Responde en español.

Respuesta:
"""
    
    # 4. Gemini genera respuesta
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.0
        )
    )
    
    return {
        "query": query,
        "retrieved_results": retrieved_results,
        "context": context,
        "answer": response.text
    }

### Probar RAG con Gemini

### Pregunta 1: ingredientes

In [61]:
result_ingredients = rag_answer_with_gemini(
    "What ingredients are needed for beer can chicken?",
    top_k=5
)

print(result_ingredients["answer"])

Aquí tienes los ingredientes para varias recetas de "Beer Can Chicken" y similares, según el contexto proporcionado:

**Para la receta "Beer Can Chicken":**
*   ⅓ taza de azúcar morena
*   2 cucharadas de chile en polvo
*   2 cucharadas de pimentón
*   2 cucharaditas de mostaza seca
*   ½ cucharadita de sal
*   ¼ cucharadita de pimienta negra molida
*   ½ (lata de 12 onzas líquidas) de cerveza
*   1 (pollo entero de 3 libras)

**Para la receta "Best Beer Can Chicken":**
*   2 tazas de virutas de madera de cerezo
*   ½ taza de azúcar morena oscura
*   ½ taza de sal
*   ½ taza de pimentón
*   ¼ taza de pimienta negra molida
*   1 cucharadita de pimienta de cayena
*   2 (latas de 12 onzas líquidas) de cerveza, medio llenas
*   2 (pollos enteros de 3 libras)
*   ¼ taza de aceite vegetal

**Para la receta "Beer Butt Chicken":**
*   1 taza de mantequilla, dividida
*   2 cucharadas de sal de ajo, dividida
*   2 cucharadas de pimentón, dividido
*   sal y pimienta al gusto
*   1 (lata de 12 onz

### Gemini respondió usando estos documentos recuperados del corpus

In [62]:
pd.DataFrame([
    {
        "rank": r["rank"],
        "score": r["score"],
        "title": r["title"],
        "section": r["section"],
        "source": r["source"]
    }
    for r in result_ingredients["retrieved_results"]
])

,rank,score,title,section,source
0,1,0.622726,Beer Can Chicken,ingredients,https://www.allrecipes.com/recipe/214618/beer-...
1,2,0.524361,Best Beer Can Chicken,ingredients,https://www.allrecipes.com/recipe/214619/bbq-b...
2,3,0.488954,Beer Butt Chicken,ingredients,https://www.allrecipes.com/recipe/14531/beer-b...
3,4,0.450395,Smoked Beer Butt Chicken,ingredients,https://www.allrecipes.com/recipe/222936/smoke...
4,5,0.439787,The Best Beer Can Chicken Ever,ingredients,https://www.allrecipes.com/recipe/228070/the-b...


### Pregunta 2: preparación

In [63]:
result_instructions = rag_answer_with_gemini(
    "How do I cook smoked whole chicken?",
    top_k=5
)

print(result_instructions["answer"])

Para cocinar "Smoked Whole Chicken", sigue estos pasos:

1.  Precalienta un ahumador a 250 grados F (120 grados C).
2.  Mezcla pimentón, chile en polvo, cayena, pimienta negra, tomillo, ajo en polvo, cebolla en polvo y sal en un tazón grande con un tenedor hasta que estén bien combinados.
3.  Deshuesa el pollo (spatchcock) colocándolo con la pechuga hacia abajo sobre una superficie de trabajo. Comenzando por la parte de la cola, corta a lo largo de ambos lados de la columna vertebral con tijeras de cocina. Retira la columna vertebral. Sujetando ambos lados del pollo, ábrelo como un libro. Gira la pechuga hacia arriba. Presiona hacia abajo en cada lado de la pechuga con las manos hasta que escuches un crujido; presiona para aplanar. Frota el pollo por todas partes con la mezcla de condimentos.
4.  Ahúma el pollo en el ahumador precalentado hasta que un termómetro de lectura instantánea insertado en la parte más gruesa del muslo, cerca del hueso, marque 140 grados F (60 grados C), aproxi

### Gemini respondió usando estos documentos recuperados del corpus

In [64]:
pd.DataFrame([
    {
        "rank": r["rank"],
        "score": r["score"],
        "title": r["title"],
        "section": r["section"],
        "source": r["source"]
    }
    for r in result_instructions["retrieved_results"]
])

,rank,score,title,section,source
0,1,0.552110,Smoked Whole Chicken,description,https://www.allrecipes.com/recipe/281255/smoke...
1,2,0.470552,Smoked Whole Chicken,instructions,https://www.allrecipes.com/recipe/281255/smoke...
2,3,0.408010,Drunk Chicken,instructions,https://www.allrecipes.com/recipe/19944/drunk-...
3,4,0.406172,Rotisserie Chicken,instructions,..\data\rotisserie-chicken.html
4,5,0.371478,Smoked Beer Butt Chicken,instructions,https://www.allrecipes.com/recipe/222936/smoke...


### Pregunta 3: nutrición

In [65]:
result_nutrition = rag_answer_with_gemini(
    "What is the nutrition information for rotisserie chicken?",
    top_k=5
)

print(result_nutrition["answer"])

Aquí tienes la información nutricional para la receta "Rotisserie Chicken":

*   **Grasa Total:** 25g
*   **Grasa Saturada:** 10g
*   **Colesterol:** 117mg
*   **Sodio:** 1311mg
*   **Carbohidratos Totales:** 1g
*   **Fibra Dietética:** 1g
*   **Azúcares Totales:** 0g
*   **Proteína:** 31g
*   **Vitamina C:** 1mg
*   **Calcio:** 22mg
*   **Hierro:** 2mg
*   **Potasio:** 302mg


### Gemini respondió usando estos documentos recuperados del corpus

In [66]:
pd.DataFrame([
    {
        "rank": r["rank"],
        "score": r["score"],
        "title": r["title"],
        "section": r["section"],
        "source": r["source"]
    }
    for r in result_nutrition["retrieved_results"]
])

,rank,score,title,section,source
0,1,0.616911,Rotisserie Chicken,instructions,..\data\rotisserie-chicken.html
1,2,0.598509,Rotisserie Chicken,nutrition,..\data\rotisserie-chicken.html
2,3,0.547589,Rotisserie Chicken,description,..\data\rotisserie-chicken.html
3,4,0.374087,Rotisserie Chicken,ingredients,..\data\rotisserie-chicken.html
4,5,0.357943,Drunk Chicken,nutrition,https://www.allrecipes.com/recipe/19944/drunk-...


## Conclusión

En esta práctica se implementó un proceso de web scraping y crawling sobre recetas de AllRecipes. Primero se cargó una receta local en formato HTML y se analizaron sus etiquetas principales con BeautifulSoup. Luego se extrajeron datos como título, descripción, ingredientes, instrucciones, información nutricional y enlaces relacionados.

Después, se filtraron únicamente URLs correspondientes a recetas reales y se descargaron varias páginas relacionadas para construir un corpus de 17 recetas. Cada receta fue almacenada con sus datos estructurados y guardada en formato JSON y CSV.

Finalmente, se implementó un flujo RAG usando el corpus obtenido. Para la etapa de recuperación se utilizó TF-IDF y similitud coseno, permitiendo seleccionar los documentos más relevantes según la pregunta del usuario. Posteriormente, el contexto recuperado fue enviado a Gemini, que generó respuestas basadas únicamente en la información del corpus.

Este flujo demuestra cómo transformar contenido web no estructurado en un corpus útil para recuperación de información y generación de respuestas asistidas por inteligencia artificial.